In [ ]:
import pandas as pd
import os
import plotly.express as px
from dash import Dash, html, dcc, Input, Output, callback

# Data loading and preparation code
def load_data():
    # Directory where the CSV files are stored
    directory = 'Data-Science-Project-WS2425/data/tempo'
    
    # List all CSV files from the directory
    csv_files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    
    # List to save all dataframes
    df_list = []
    
    # Read all files and extract the years
    for file in csv_files:
        df = pd.read_csv(os.path.join(directory, file))
        # Extract year from file name
        year = int(file.split('_')[1])
        # Add Year as a new column
        df['Year'] = year
        
        # Add dataframe to list
        df_list.append(df)
    
    # Join all dataframes
    df_all_years_tempo = pd.concat(df_list, ignore_index=True)
    
    # Below function has been generated/modified with the help of ChatGPT.
    # Fix "Tempo" Column: Remove spaces, "BPM", and handle errors
    df_all_years_tempo["Tempo"] = (
        df_all_years_tempo["Tempo"]
        .astype(str)  # Ensure all values are strings
        .str.strip()  # Remove leading/trailing spaces
        .str.replace(r"\s*BPM", "", regex=True)  # Remove "BPM"
        .apply(pd.to_numeric, errors="coerce")  # Convert to float, set errors to NaN
    )
    
    # Drop NaN values (optional, depending on your data)
    df_all_years_tempo = df_all_years_tempo.dropna(subset=["Tempo"])
    
    # Convert to int
    df_all_years_tempo["Tempo"] = df_all_years_tempo["Tempo"].astype(int)
    
    # Group by year and calculate the mean
    df_grouped = df_all_years_tempo.groupby("Year", as_index=False)["Tempo"].mean()
    
    # Remove "" from titles
    df_all_years_tempo["Title"] = df_all_years_tempo["Title"].str.replace(r'^"|"$', '', regex=True)
    
    # Sort by year
    df_all_years_tempo = df_all_years_tempo.sort_values(by="Year")
    
    return df_all_years_tempo, df_grouped

# Create the Dash app
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = Dash(__name__, external_stylesheets=external_stylesheets)

# Load data
df_all_years_tempo, df_grouped = load_data()

app.layout = html.Div([
    html.H1("Average Tempo (BPM) Of Songs In The Last 20 Years"),
    
    # Year range slider
    dcc.RangeSlider(
        min=2005,
        max=2024,
        step=1,
        value=[2005, 2024],
        marks={str(year): str(year) for year in sorted(df_all_years_tempo["Year"].unique())},
        id='year-range-slider'
    ),
    
    # This contains the scatterplot
    html.Div(id='scatter-plot-container')
])

@callback(
    Output('scatter-plot-container', 'children'),
    Input('year-range-slider', 'value')
)
def update_graph(years_range):
    # Filter data for selected years
    filtered_df = df_all_years_tempo[
        (df_all_years_tempo['Year'] >= years_range[0]) & 
        (df_all_years_tempo['Year'] <= years_range[1])
    ]
    
    # Calculate grouped data for the filtered range
    filtered_grouped = filtered_df.groupby('Year', as_index=False)['Tempo'].mean()
    

    # Create the scatter plot 
    fig = px.scatter(filtered_df, x="Year", y="Tempo",
                    title="Average Tempo (BPM) Of Songs In The Last 20 Years",
                    labels={"Tempo": "Tempo (BPM)", "Year": "Year"},
                    opacity=0.3)
    
    # Line for the average tempo values
    fig.add_scatter(x=filtered_grouped["Year"], y=filtered_grouped["Tempo"], 
                    mode="lines", name="Average", line=dict(width=2))
    
    # Update x-axes type
    fig.update_xaxes(type='category')
    
    # Makes sure y-axis shows BPM range properly
    fig.update_yaxes(
        title_text="Tempo (BPM)",
        range=[filtered_df['Tempo'].min() - 10, filtered_df['Tempo'].max() + 10]
    )
    
    
    return dcc.Graph(figure=fig)

if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
import plotly.express as px
import pandas as pd
import os
import numpy as np
from dash import Dash, html, dcc, Input, Output

# Directory in which the duration files are saved
directory = 'Data-Science-Project-WS2425/data/durations'

# List all CSV-files from the directory
csv_files = [f for f in os.listdir(directory) if f.endswith('.csv')]

# List to save all dataframes
df_list = []

# Read all Files and extract the years
for file in csv_files:
    df = pd.read_csv(os.path.join(directory, file))
    # Extract the year from the file name    
    year = int(file.split('_')[1])  
    df['Year'] = year
    df_list.append(df)

# Concatenate all dataframes into one,    
df_all_years = pd.concat(df_list, ignore_index=True)

# Below function has been generated/modified with the help of ChatGPT.
# Function to remove outliers using IQR method
def remove_outliers(df, column):
    df_copy = df.copy()
    Q1 = df_copy[column].quantile(0.25)
    Q3 = df_copy[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Filter out the rows that fall outside the valid range
    return df_copy[(df_copy[column] >= lower_bound) & (df_copy[column] <= upper_bound)]

# Helper function to convert minutes to mm:ss format
def format_duration(minutes):
    total_seconds = int(minutes * 60)  
    mm = total_seconds // 60  
    ss = total_seconds % 60   
    return f"{mm}:{ss:02d}"

# Initialize Dash app
app = Dash(__name__, external_stylesheets=['https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css'])

# Define the layout
app.layout = html.Div([
    
    html.Div([
        html.Label('Show Outliers', style={'marginRight': '10px'}),
        dcc.Checklist(
            id='outlier-toggle',
            options=[{'label': '', 'value': 'show'}], # Option to show or hide outliers
            value=['show'], # Default value is to show outliers
            inline=True
        )
    ], style={'display': 'flex', 'alignItems': 'center', 'margin': '20px 0'}),
    
    dcc.Graph(id='duration-chart')
])

# Define callback to update the chart based on the user's selection
@app.callback(
    Output('duration-chart', 'figure'),
    Input('outlier-toggle', 'value')
)
def update_chart(show_outliers):
    df_original = df_all_years  # Original dataframe 
    
    # If outliers should be shown, use the original data
    if 'show' in show_outliers:
        df_to_use = df_original
        title = "Average Song Durations Over 20 Years (With Outliers)"
    else:
        df_to_use = remove_outliers(df_original, "Duration_min")
        title = "Average Song Durations Over 20 Years (Without Outliers)"
    
    # Group data by year and calculate the average duration for each year
    df_grouped = df_to_use.groupby("Year", as_index=False)["Duration_min"].mean()
    
    # Add a column for formatted duration
    df_grouped['Duration_formatted'] = df_grouped['Duration_min'].apply(format_duration)
    
    # Define the maximum value for the y-axis
    y_max = 5  
    y_ticks = list(range(0, y_max))  
    y_labels = [format_duration(y) for y in y_ticks]  
    
    # Create a bar chart with custom hover template
    fig = px.bar(df_grouped, x="Year", y="Duration_min", title=title)
    
    # Customize hover info to show both decimal and mm:ss formats
    fig.update_traces(
        hovertemplate='<b>Year:</b> %{x}<br>' +
                      '<b>Duration (mm:ss):</b> %{customdata}<br>' +
                      '<b>Duration (minutes):</b> %{y:.2f}<extra></extra>',
        customdata=df_grouped['Duration_formatted']
    )
    
    fig.update_layout(
        xaxis=dict(
            tickmode="array",
            tickvals=df_grouped["Year"],
            tickformat=".0f",
            rangeslider=dict(visible=True), # Enable range slider for x-axis
            type="linear",
            fixedrange=True  # Disable zooming
        ),
        yaxis=dict(
            title="Duration",
            tickvals=y_ticks,
            ticktext=y_labels,
            fixedrange=True,  
            range=[0, y_max]  
        )
    )
    
    return fig

if __name__ == '__main__':
    app.run(debug=True)

How has song duration changed over the past 20 years?

The visualization "Average Song Durations Over 20 Years (With Outliers)," shows an overall downward trend. The average song length has dropped from around 4:00 minutes in 2005 to about 3:20 minutes in 2024, showing a decrease of roughly 17%. 2005 to 2009 is as relatively stable period, with durations averaging around 4:00 minutes. 2010 to 2017 a gradual but steady decline down to about 3:45 minutes is noticable. 2018 to 2024 shows a sharper drop, followed by stabilization at around 3:15-3:30 minutes. The most noticeable drop happened between 2017 and 2019, suggesting a major shift in the music industry or cultural trends during that time and a possible correlation between the durations and use of streaming platforms or social media. Since 2019, average song lengths seem to have settled at a lower range of around 3:15-3:20 minutes.

(The visualizations without outlier also shows this trend and doesn't change much about the result.)